# Bengaluru Road Network — Real OSMnx Data
### Chinnaswamy Stadium Zone · Flipkart GRID 2026

This notebook downloads the **real road network** from OpenStreetMap around Chinnaswamy Stadium (2 km radius) and visualises it.

**What you'll see:**
1. Raw road network — every drivable junction and segment within 2 km
2. Betweenness centrality heatmap — which junctions control the most flow (officer deployment targets)
3. Interactive folium map — pan/zoom the real Bengaluru streets

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install osmnx>=2.0 folium --quiet

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────────────────
VENUE_NAME   = "Chinnaswamy Stadium, Bengaluru"
VENUE_LAT    = 12.9788
VENUE_LNG    = 77.5996
ZONE_RADIUS_M = 2000   # study zone radius in metres

In [ ]:
# ── Cell 3: Download real road network from OpenStreetMap ─────────────────
import osmnx as ox
import networkx as nx
import numpy as np
from collections import Counter

print(f"Downloading drive network within {ZONE_RADIUS_M}m of {VENUE_NAME}...")
G_multi = ox.graph_from_point(
    (VENUE_LAT, VENUE_LNG),
    dist=ZONE_RADIUS_M,
    network_type="drive"
)

# Normalise coordinates: OSMnx stores lat in 'y', lng in 'x'
for _, d in G_multi.nodes(data=True):
    d["lat"] = d["y"]
    d["lng"] = d["x"]

# Convert MultiDiGraph → DiGraph (keeps shortest parallel edge)
G = ox.convert.to_digraph(G_multi, weight="length")
for _, d in G.nodes(data=True):
    if "lat" not in d:
        d["lat"] = d.get("y", 0.0)
        d["lng"] = d.get("x", 0.0)

print(f"\n{'='*50}")
print(f" ROAD NETWORK SUMMARY")
print(f"{'='*50}")
print(f" Junctions (nodes) : {G.number_of_nodes():>6,}")
print(f" Road segments     : {G.number_of_edges():>6,}")

In [ ]:
# ── Cell 4: Detailed network stats ────────────────────────────────────────

def _parse_lanes(x, default=2):
    if x is None: return default
    if isinstance(x, list):
        try: return max(int(str(v).split(";")[0].strip()) for v in x if v)
        except: return default
    try: return max(1, int(str(x).split(";")[0].strip()))
    except: return default

def _parse_speed(x, default=40.0):
    if x is None: return float(default)
    if isinstance(x, list): x = x[0] if x else None
    if x is None: return float(default)
    s = str(x).strip().lower()
    if "mph" in s:
        try: return round(float(s.replace("mph","").strip()) * 1.60934, 1)
        except: return float(default)
    try: return float(s.replace("km/h","").replace("kph","").strip())
    except: return float(default)

def _first(x):
    if isinstance(x, list): return x[0] if x else "unnamed"
    return x if x else "unnamed"

# Build seg_info lookup
seg_info = {}
for u, v, d in G.edges(data=True):
    lanes = _parse_lanes(d.get("lanes"))
    ffs   = _parse_speed(d.get("maxspeed"))
    seg_info[f"{u}__{v}"] = {
        "u": u, "v": v,
        "road":     _first(d.get("name", "unnamed")),
        "lanes":    lanes,
        "ffs":      ffs,
        "capacity": lanes * ffs,
        "length_m": float(d.get("length", 0.0)),
    }

roads  = [v["road"]  for v in seg_info.values()]
lanes  = [v["lanes"] for v in seg_info.values()]
speeds = [v["ffs"]   for v in seg_info.values()]

print("Top 15 roads by segment count:")
for road, cnt in Counter(roads).most_common(15):
    print(f"  {road:<35} {cnt:>4} segments")

print("\nLane distribution:")
for l, cnt in sorted(Counter(lanes).items()):
    print(f"  {l} lanes: {cnt:,} segments")

print(f"\nFree-flow speed (km/h):")
print(f"  min={min(speeds):.0f}  max={max(speeds):.0f}  "
      f"mean={np.mean(speeds):.1f}  median={np.median(speeds):.1f}")

In [ ]:
# ── Cell 5: OSMnx built-in plot — clean road map ──────────────────────────
import matplotlib.pyplot as plt

fig, ax = ox.plot_graph(
    G_multi,
    figsize=(12, 12),
    bgcolor="#0d1117",
    node_color="#3a7abf",
    node_size=3,
    edge_color="#3a7abf",
    edge_linewidth=0.5,
    edge_alpha=0.6,
    show=False,
    close=False
)

# Mark venue
ax.scatter([VENUE_LNG], [VENUE_LAT], s=300, c="#ff4444",
           zorder=10, marker="*", label="Chinnaswamy Stadium")

# Zone circle (approx)
theta = np.linspace(0, 2 * np.pi, 300)
r_lat = ZONE_RADIUS_M / 111_000
r_lng = ZONE_RADIUS_M / (111_000 * np.cos(np.radians(VENUE_LAT)))
ax.plot(VENUE_LNG + r_lng * np.cos(theta),
        VENUE_LAT + r_lat * np.sin(theta),
        "--", color="#ffaa00", lw=1.5, alpha=0.8, label="2 km study zone")

ax.legend(facecolor="#1c2333", edgecolor="#3a7abf", labelcolor="white", fontsize=11)
ax.set_title(f"Real OSM Road Network — {VENUE_NAME}\n"
             f"{G.number_of_nodes():,} junctions · {G.number_of_edges():,} segments",
             color="white", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6: Betweenness centrality — find traffic chokepoints ─────────────
print("Computing betweenness centrality (may take ~30s on 2700+ nodes)...")
bc = nx.betweenness_centrality(G)

# Attach centrality to nodes in the graph
for n, score in bc.items():
    G.nodes[n]["bc"] = score

# Top 10 bottleneck junctions
print("\nTop 10 traffic chokepoints:")
print(f"{'Rank':<5} {'Node ID':<15} {'Centrality':<12} {'Lat':<10} {'Lng':<10}")
print("-" * 55)
for rank, (node, score) in enumerate(sorted(bc.items(), key=lambda x: -x[1])[:10], 1):
    lat = G.nodes[node]["lat"]
    lng = G.nodes[node]["lng"]
    print(f"{rank:<5} {node:<15} {score:<12.4f} {lat:<10.4f} {lng:<10.4f}")

In [ ]:
# ── Cell 7: Centrality heatmap plot ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 9), facecolor="#0d1117")

node_pos = {n: (d["lng"], d["lat"]) for n, d in G.nodes(data=True)}
node_list = list(G.nodes())
centrality_vals = [bc[n] for n in node_list]
node_x = [G.nodes[n]["lng"] for n in node_list]
node_y = [G.nodes[n]["lat"] for n in node_list]

# LEFT: road skeleton
ax1 = axes[0]
ax1.set_facecolor("#0d1117")
nx.draw_networkx_edges(G, pos=node_pos, ax=ax1,
                       edge_color="#3a7abf", alpha=0.35, width=0.4, arrows=False)
ax1.scatter([VENUE_LNG], [VENUE_LAT], s=250, c="#ff4444",
            zorder=10, marker="*", label="Chinnaswamy Stadium")
ax1.plot(VENUE_LNG + r_lng * np.cos(theta),
         VENUE_LAT + r_lat * np.sin(theta),
         "--", color="#ffaa00", lw=1.2, alpha=0.7, label="2 km zone")
ax1.set_title(f"Drive Network — {G.number_of_nodes():,} junctions · {G.number_of_edges():,} segments",
              color="white", fontsize=12, pad=8)
ax1.legend(facecolor="#1c2333", edgecolor="#3a7abf", labelcolor="white", fontsize=9)
ax1.tick_params(colors="#888"); ax1.set_xlabel("Longitude", color="#888")
ax1.set_ylabel("Latitude", color="#888")
for sp in ax1.spines.values(): sp.set_edgecolor("#3a7abf")

# RIGHT: centrality heatmap
ax2 = axes[1]
ax2.set_facecolor("#0d1117")
nx.draw_networkx_edges(G, pos=node_pos, ax=ax2,
                       edge_color="#1a1a2e", alpha=0.4, width=0.3, arrows=False)
sc = ax2.scatter(node_x, node_y, c=centrality_vals, cmap="plasma",
                 s=10, alpha=0.9, zorder=5,
                 vmin=0, vmax=np.percentile(centrality_vals, 95))
cbar = fig.colorbar(sc, ax=ax2, fraction=0.03, pad=0.02)
cbar.set_label("Betweenness Centrality\n(higher = bigger chokepoint)",
               color="white", fontsize=9)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white", fontsize=7)
ax2.scatter([VENUE_LNG], [VENUE_LAT], s=300, c="#ff4444",
            zorder=10, marker="*")
ax2.set_title("Junction Bottleneck Map (Betweenness Centrality)\nBrighter = bigger traffic chokepoint → officer priority",
              color="white", fontsize=12, pad=8)
ax2.tick_params(colors="#888"); ax2.set_xlabel("Longitude", color="#888")
ax2.set_ylabel("Latitude", color="#888")
for sp in ax2.spines.values(): sp.set_edgecolor("#3a7abf")

plt.suptitle("Bengaluru Traffic Network · Chinnaswamy Zone · Real OSM Data",
             color="white", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8: Interactive Folium map — pan & zoom the real streets ──────────
import folium
from IPython.display import display

m = folium.Map(location=[VENUE_LAT, VENUE_LNG], zoom_start=15,
               tiles="CartoDB dark_matter")

# Draw road segments as polylines
for u, v, d in G.edges(data=True):
    lat_u = G.nodes[u]["lat"]; lng_u = G.nodes[u]["lng"]
    lat_v = G.nodes[v]["lat"]; lng_v = G.nodes[v]["lng"]
    folium.PolyLine(
        locations=[[lat_u, lng_u], [lat_v, lng_v]],
        color="#3a7abf", weight=1.5, opacity=0.5
    ).add_to(m)

# Colour top-20 bottleneck nodes in red/orange, rest in blue
top_nodes = set(n for n, _ in sorted(bc.items(), key=lambda x: -x[1])[:20])
for n, d in G.nodes(data=True):
    color  = "red"    if n in top_nodes else "#1a5276"
    radius = 7        if n in top_nodes else 2
    popup  = (f"Node {n}<br>Centrality: {bc[n]:.4f}<br>"
              f"({d['lat']:.4f}, {d['lng']:.4f})")
    folium.CircleMarker(
        location=[d["lat"], d["lng"]],
        radius=radius, color=color, fill=True, fill_opacity=0.8,
        popup=folium.Popup(popup, max_width=200)
    ).add_to(m)

# Venue marker
folium.Marker(
    location=[VENUE_LAT, VENUE_LNG],
    popup="Chinnaswamy Stadium",
    icon=folium.Icon(color="red", icon="star")
).add_to(m)

# Zone circle
folium.Circle(
    location=[VENUE_LAT, VENUE_LNG],
    radius=ZONE_RADIUS_M,
    color="orange", fill=False, weight=2, dash_array="10"
).add_to(m)

display(m)

In [ ]:
# ── Cell 9: Export seg_info for downstream use ────────────────────────────
import pandas as pd

seg_df = pd.DataFrame(seg_info).T.reset_index()
seg_df.columns = ["segment_id"] + list(seg_df.columns[1:])
print(f"seg_info exported: {len(seg_df):,} rows")
print(seg_df.head(10).to_string(index=False))